In [ ]:
import sys, types, warnings, glob, os, re
warnings.filterwarnings('ignore')

fake_mamba3 = types.ModuleType('mamba_ssm.modules.mamba3')
fake_mamba3.Mamba3 = type('Mamba3', (), {})
sys.modules['mamba_ssm.modules.mamba3'] = fake_mamba3

import pandas as pd, numpy as np
from transformers import AutoTokenizer
import kagglehub
print('imports OK')

In [ ]:
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('tokenizer OK')

cot_paths = glob.glob('/kaggle/input/**/problem_ids_matched.csv', recursive=True)
assert cot_paths, 'CoT csv not found'
cot_df = pd.read_csv(cot_paths[0])
print(f'CoT dataset: {len(cot_df)} rows, cols: {list(cot_df.columns)}')

In [ ]:
# Build texts with the EXACT same logic as the training kernel (cell 3).
PROMPT_SUFFIX = '\nPut your final answer inside \\boxed{}.'
texts, skipped = [], 0
for _, row in cot_df.iterrows():
    prompt = str(row['prompt']); answer = str(row['answer'])
    cot = str(row.get('generated_cot', ''))
    if not cot or cot == 'nan' or len(cot.strip()) < 5:
        skipped += 1; continue
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    user_content = prompt + PROMPT_SUFFIX
    assistant_content = f'<think>\n{cot_cleaned}\n</think>\n\\boxed{{{answer}}}'
    try:
        text = tokenizer.apply_chat_template(
            [{'role':'user','content':user_content},{'role':'assistant','content':assistant_content}],
            tokenize=False, add_generation_prompt=False)
    except Exception:
        text = f'<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n{assistant_content}<|im_end|>'
    texts.append(text)
print(f'built {len(texts)} texts, skipped {skipped} (empty/short CoT)')

lengths = np.array([len(ids) for ids in tokenizer(texts, add_special_tokens=False)['input_ids']])

print('\n=== TOKEN LENGTH DISTRIBUTION (full dataset) ===')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p:<3} = {np.percentile(lengths, p):.0f}')
print(f'  min={lengths.min()}  max={lengths.max()}  mean={lengths.mean():.0f}')

N = len(lengths)
print('\n=== TRUNCATION / FILTER TABLE ===')
print(f'{"max_len":>8} | {"complete":>9} | {"complete%":>9} | {"truncated":>9} | {"mean(kept)":>10} | {"steps/8":>8}')
for L in [2048, 3072, 4096, 6144, 8192]:
    kept = lengths[lengths <= L]; nkept = len(kept); steps = -(-nkept // 8)
    mlen = kept.mean() if nkept else 0
    print(f'{L:>8} | {nkept:>9} | {100*nkept/N:>8.1f}% | {N-nkept:>9} | {mlen:>10.0f} | {steps:>8}')
print(f'\ntotal usable examples: {N}')
print('"complete" = full sequence fits, so </think>+boxed answer is NOT cut off.')

print('\n=== ROUGH GPU-TIME ESTIMATE (12h kernel limit, ~580s model-load overhead) ===')
SEC_PER_STEP_AT_2048 = 53  # measured in v21
for L in [2048, 3072, 4096, 6144]:
    kept = lengths[lengths <= L]; nkept = len(kept); steps = -(-nkept // 8)
    mlen = kept.mean() if nkept else 0
    est_h = (steps * SEC_PER_STEP_AT_2048 * (mlen / 2048) + 580) / 3600
    print(f'  max_len={L}: filter->{nkept} ex, {steps} steps, ~{est_h:.1f}h '
          f'(linear est; eager-attn quadratic term makes real time HIGHER)')